# Chapitre 7 — Évaluation du RAG

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-07-evaluation/07_evaluation.ipynb)

Les métriques de retrieval sont locales et reproductibles. Le juge OpenAI est facultatif et sert uniquement aux critères sémantiques.

## Ressources utiles

- [OpenAI Docs — bonnes pratiques d'évaluation](https://developers.openai.com/api/docs/guides/evaluation-best-practices)
- [OpenAI Developers — ressources sur les évaluations](https://developers.openai.com/learn/evals)
- [scikit-learn — nDCG](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ndcg_score.html)
- [RAGAS — métriques](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)

## 0. Préparer Colab ou Jupyter

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("Environnement du chapitre 7 prêt :", Path.cwd())


## Configuration facultative du juge OpenAI

In [ ]:
# @title Activer le juge OpenAI pour les exemples 3 et 4
UTILISER_OPENAI = False # @param {type:"boolean"}

if UTILISER_OPENAI:
    import os
    import subprocess
    import sys
    from getpass import getpass

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai]"], check=True)
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
    if not os.getenv("OPENAI_MODEL"):
        os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()
    print("Le juge OpenAI est activé.")
else:
    print("Juge OpenAI désactivé. Les métriques de retrieval restent exécutables.")


## 1. Métriques de rang

Mesurer séparément présence, précision, rappel et position du premier résultat utile.

Script correspondant : [`01_metriques_rang.py`](examples/01_metriques_rang.py)

In [ ]:
# ruff: noqa: F811
"""Calculer les métriques de rang d'un retriever sans appel de modèle."""

from __future__ import annotations

import math


def evaluer_classement(
    recuperes: list[str],
    pertinents: set[str],
    k: int = 5,
) -> dict[str, float]:
    """Calcule Hit Rate, Precision@k, Recall@k, MRR et nDCG binaire."""
    if k <= 0:
        raise ValueError("k doit être strictement positif")
    tetes = recuperes[:k]
    trouves = [document for document in tetes if document in pertinents]
    hit = float(bool(trouves))
    precision = len(trouves) / k
    rappel = len(set(trouves)) / len(pertinents) if pertinents else 0.0
    mrr = next(
        (1.0 / rang for rang, document in enumerate(tetes, start=1) if document in pertinents),
        0.0,
    )
    dcg = sum(
        1.0 / math.log2(rang + 1)
        for rang, document in enumerate(tetes, start=1)
        if document in pertinents
    )
    ideal = sum(
        1.0 / math.log2(rang + 1)
        for rang in range(1, min(len(pertinents), k) + 1)
    )
    return {
        "hit_rate": hit,
        "precision_at_k": precision,
        "recall_at_k": rappel,
        "mrr": mrr,
        "ndcg": dcg / ideal if ideal else 0.0,
    }


if __name__ == "__main__":
    resultat = evaluer_classement(
        ["chunk_8", "chunk_2", "chunk_5", "chunk_1"],
        {"chunk_2", "chunk_1"},
        k=3,
    )
    print({nom: round(valeur, 3) for nom, valeur in resultat.items()})


## 2. nDCG gradué

Récompenser davantage les passages hautement pertinents placés en tête.

Script correspondant : [`02_ndcg_gradue.py`](examples/02_ndcg_gradue.py)

In [ ]:
# ruff: noqa: F811
"""Calculer le nDCG avec des annotations de pertinence graduées."""

from __future__ import annotations

import math


def gain(note: int, exponentiel: bool = True) -> float:
    if note < 0:
        raise ValueError("Une note de pertinence ne peut pas être négative")
    return float(2**note - 1 if exponentiel else note)


def ndcg_gradue(
    recuperes: list[str],
    pertinence: dict[str, int],
    k: int = 5,
    *,
    gain_exponentiel: bool = True,
) -> float:
    if k <= 0:
        raise ValueError("k doit être strictement positif")
    dcg = sum(
        gain(pertinence.get(document, 0), gain_exponentiel) / math.log2(rang + 1)
        for rang, document in enumerate(recuperes[:k], start=1)
    )
    meilleures = sorted(pertinence.values(), reverse=True)[:k]
    idcg = sum(
        gain(note, gain_exponentiel) / math.log2(rang + 1)
        for rang, note in enumerate(meilleures, start=1)
    )
    return dcg / idcg if idcg else 0.0


if __name__ == "__main__":
    notes = {"chunk_1": 3, "chunk_2": 2, "chunk_3": 1, "chunk_4": 0}
    print(round(ndcg_gradue(["chunk_3", "chunk_1", "chunk_4"], notes, k=3), 3))


## 3. Fidélité détaillée

Décomposer la réponse puis vérifier chaque affirmation avec un juge OpenAI.

Script correspondant : [`03_fidelite_maison.py`](examples/03_fidelite_maison.py)

In [ ]:
# ruff: noqa: F811
"""Mesurer la fidélité d'une réponse avec un juge OpenAI injectable."""

from __future__ import annotations

from rag_en_pratique.prompting import generer, openai_configure


def calculer_faithfulness(
    reponse: str,
    contexte: str,
    *,
    client=None,
    model: str | None = None,
) -> dict[str, object]:
    """Décompose la réponse, puis vérifie chaque affirmation contre le contexte."""
    brut = generer(
        "Décompose le texte en faits vérifiables, un par ligne, sans numérotation ni ajout.",
        reponse,
        client=client,
        model=model,
    )
    affirmations = [ligne.strip(" -•\t") for ligne in brut.splitlines() if ligne.strip()]
    if not affirmations:
        return {"score": 1.0, "detail": [], "total": 0}

    detail = []
    for affirmation in affirmations:
        verdict = generer(
            "Réponds par un seul mot : ETAYEE, PARTIELLE ou ABSENTE.",
            (
                f"PASSAGES :\n{contexte}\n\nAFFIRMATION : {affirmation}\n\n"
                "Peut-elle être déduite des passages ? Une information vraie mais absente est ABSENTE."
            ),
            client=client,
            model=model,
        ).upper()
        etiquette = verdict.split()[0] if verdict else "ABSENTE"
        poids = {"ETAYEE": 1.0, "ÉTAYÉE": 1.0, "PARTIELLE": 0.5}.get(etiquette, 0.0)
        detail.append({"affirmation": affirmation, "verdict": verdict, "poids": poids})
    return {
        "score": sum(item["poids"] for item in detail) / len(detail),
        "detail": detail,
        "total": len(detail),
    }


if __name__ == "__main__":
    if openai_configure():
        print(calculer_faithfulness("La garantie dure 24 mois.", "Garantie : 24 mois."))
    else:
        print("Exemple prêt : configurez OPENAI_API_KEY et OPENAI_MODEL.")


## 4. Campagne complète

Conserver les scores par cas, leurs moyennes, les métadonnées et un diagnostic.

Script correspondant : [`04_campagne_evaluation.py`](examples/04_campagne_evaluation.py)

In [ ]:
# ruff: noqa: F811
"""Exécuter une campagne d'évaluation RAG et produire un diagnostic."""

from __future__ import annotations

import statistics
from collections.abc import Callable
from typing import Any

from rag_en_pratique.prompting import generer, openai_configure

SEUILS = {
    "faithfulness": 0.80,
    "answer_relevance": 0.75,
    "contextual_precision": 0.65,
    "contextual_recall": 0.70,
}


def diagnostiquer(moyennes: dict[str, float], seuils: dict[str, float] | None = None) -> str:
    s = seuils or SEUILS
    bas = {nom for nom, seuil in s.items() if moyennes.get(nom, 0.0) < seuil}
    if not bas:
        return "Système sain. Surveiller sans intervenir."
    if "faithfulness" in bas and "answer_relevance" not in bas:
        return "Hallucination à la génération : renforcer l'ancrage du prompt."
    if "answer_relevance" in bas and "contextual_recall" not in bas:
        return "Échec à la génération : revoir le prompt, le format ou le modèle."
    if {"contextual_precision", "contextual_recall"} <= bas:
        return "Échec au retrieval : vérifier le corpus avant d'optimiser le retriever."
    if bas == {"contextual_precision"}:
        return "Retriever bruyant : reclasser, réduire k ou relever le seuil."
    if bas == {"contextual_recall"}:
        return "Retriever incomplet : recherche hybride, k plus grand ou nouveau découpage."
    if len(bas) >= 3:
        return "Défaut systémique : reprendre depuis l'ingestion."
    return f"Configuration mixte, métriques basses : {sorted(bas)}"


def score_juge(
    critere: str,
    question: str,
    contenu: str,
    *,
    client=None,
    model: str | None = None,
) -> float:
    texte = generer(
        "Évalue selon le critère demandé. Réponds uniquement par un nombre entre 0 et 1.",
        f"CRITÈRE : {critere}\nQUESTION : {question}\nCONTENU :\n{contenu}",
        client=client,
        model=model,
    )
    try:
        return max(0.0, min(1.0, float(texte.replace(",", "."))))
    except ValueError as erreur:
        raise ValueError(f"Score du juge invalide : {texte!r}") from erreur


def rappel_contextuel(attendus: set[str], sources: list[Any]) -> float:
    if not attendus:
        return 1.0
    recuperes = {
        str(source.metadata.get("id", source.metadata.get("source", ""))) for source in sources
    }
    return len(attendus & recuperes) / len(attendus)


def lancer_campagne(
    jeu: list[dict[str, Any]],
    systeme,
    metadonnees: dict[str, str],
    *,
    faithfulness: Callable[[str, str], float],
    answer_relevance: Callable[[str, str], float],
    contextual_precision: Callable[[str, list[Any]], float],
) -> dict[str, object]:
    par_metrique: dict[str, list[float]] = {nom: [] for nom in SEUILS}
    par_cas = []
    for cas in jeu:
        sortie = systeme.repondre(cas["question"])
        sources = list(sortie["sources"])
        contexte = "\n\n".join(source.page_content for source in sources)
        scores = {
            "faithfulness": faithfulness(sortie["reponse"], contexte),
            "answer_relevance": answer_relevance(cas["question"], sortie["reponse"]),
            "contextual_precision": contextual_precision(cas["question"], sources),
            "contextual_recall": rappel_contextuel(set(cas["passages_attendus"]), sources),
        }
        for nom, valeur in scores.items():
            par_metrique[nom].append(valeur)
        par_cas.append(
            {**scores, "question": cas["question"], "type": cas.get("type"), "sujet": cas.get("sujet")}
        )
    moyennes = {
        nom: round(statistics.mean(valeurs), 3)
        for nom, valeurs in par_metrique.items()
        if valeurs
    }
    return {
        "moyennes": moyennes,
        "diagnostic": diagnostiquer(moyennes),
        "par_cas": par_cas,
        "metadonnees": metadonnees,
    }


if __name__ == "__main__":
    if openai_configure():
        print("Injectez votre système RAG et les trois fonctions de mesure dans lancer_campagne().")
    else:
        print("La campagne est prête ; les métriques locales n'exigent aucune clé.")


## Bilan

Une moyenne seule ne suffit pas : conservez le détail par question, la version du jeu, le modèle juge et les seuils. Calibrez les juges automatiques sur des annotations humaines.